# Find Candidate Induction Heads

Using the head embedding distances from notebook 03/04, find heads in other models
(pythia-1b, gemma-2b, etc.) that are nearby known GPT-2 small induction heads.

These candidates are then tested via ablation in notebook 06.

In [ ]:
from collections import Counter
from pathlib import Path

import matplotlib.pyplot as plt
import polars as pl
import numpy as np

# attention-motifs
from attention_motifs.attnpedia.attnpedia import AttentionPedia
from attention_motifs.features.analysis import DistanceTensorResult
from attention_motifs.ablation.candidates import (
	DistanceCandidates,
	get_known_induction_heads,
	find_candidate_induction_heads,
	get_control_heads,
)

In [ ]:
# config
pl.Config.set_tbl_rows(20)
PATH_BASE: Path = Path("../data/")

# Load distances and known heads

In [ ]:
# load head distances
HEAD_DISTS: DistanceTensorResult = DistanceTensorResult.read_raw(
	PATH_BASE / "features" / "head_dists_raw",
)
print(f"Loaded distances for {HEAD_DISTS.n_heads} heads")
print(f"Models: {set(h.split(':')[0] for h in HEAD_DISTS.cls_values)}")

In [ ]:
# load known induction heads
ATTNPEDIA: AttentionPedia = AttentionPedia()
KNOWN_INDUCTION: list[str] = get_known_induction_heads(ATTNPEDIA)
print(f"Known induction heads ({len(KNOWN_INDUCTION)}):")
for head in KNOWN_INDUCTION:
	print(f"  {head}")

# Find candidate induction heads in other models

In [ ]:
# find candidates based on embedding proximity
CANDIDATES: DistanceCandidates = find_candidate_induction_heads(
	HEAD_DISTS,
	reference_heads=KNOWN_INDUCTION,
	k_neighbors=20,
	exclude_reference_model=True,
	score_method="frequency",
)

print(f"Reference heads used: {len(CANDIDATES.reference_heads)}")
print(f"Models with candidates: {list(CANDIDATES.candidates_by_model.keys())}")

In [ ]:
# show top candidates per model
for model, candidates in CANDIDATES.candidates_by_model.items():
	print(f"\n{model}:")
	for head, score in candidates[:10]:
		print(f"  {head}: {score:.3f}")

In [ ]:
# convert to DataFrame for analysis
CANDIDATES_DF: pl.DataFrame = CANDIDATES.to_dataframe()
CANDIDATES_DF

# Visualize candidate distribution

In [ ]:
# plot score distribution by model
fig, axes = plt.subplots(1, 3, figsize=(18, 5))

# histogram of scores
score_bins = np.linspace(0, float(CANDIDATES_DF["score"].max()), 21)  # type: ignore[arg-type]  # ty: ignore[invalid-argument-type]
for model in CANDIDATES.candidates_by_model.keys():
	model_df = CANDIDATES_DF.filter(pl.col("model") == model)
	counts, edges = np.histogram(model_df["score"].to_numpy(), bins=score_bins)
	centers = (edges[:-1] + edges[1:]) / 2
	axes[0].plot(centers, counts, "-o", label=model)

axes[0].set_xlabel("Score")
axes[0].set_ylabel("Count")
axes[0].set_title("Candidate Score Distribution by Model")
axes[0].legend()

# layer distribution of top candidates
TOP_N: int = 10
layer_bins = np.arange(0, 21)
for model in CANDIDATES.candidates_by_model.keys():
	top_df = CANDIDATES_DF.filter(pl.col("model") == model).head(TOP_N)
	layers = top_df["layer"].to_numpy()
	counts, edges = np.histogram(layers, bins=layer_bins)
	centers = (edges[:-1] + edges[1:]) / 2
	axes[1].plot(centers, counts, "-o", label=model)

axes[1].set_xlabel("Layer")
axes[1].set_ylabel("Count")
axes[1].set_title(f"Layer Distribution of Top {TOP_N} Candidates")
axes[1].legend()

# layer depth distribution (normalized by model depth)
depth_bins = np.linspace(0, 1, 11)
for model in CANDIDATES.candidates_by_model.keys():
	top_df = CANDIDATES_DF.filter(pl.col("model") == model).head(TOP_N)
	layer_depths = top_df["layer_depth"].to_numpy()
	counts, edges = np.histogram(layer_depths, bins=depth_bins)
	centers = (edges[:-1] + edges[1:]) / 2
	axes[2].plot(centers, counts, "-o", label=model)

# add known gpt2-small induction heads (dashed)
GPT2_SMALL_N_LAYERS = 12
known_layers = np.array([int(h.split(":")[1][1:]) for h in KNOWN_INDUCTION])
known_depths = known_layers / GPT2_SMALL_N_LAYERS
counts, edges = np.histogram(known_depths, bins=depth_bins)
centers = (edges[:-1] + edges[1:]) / 2
axes[2].plot(centers, counts, "--o", label="gpt2-small (known)", color="black")

axes[2].set_xlabel("Layer Depth (layer / num_layers)")
axes[2].set_ylabel("Count")
axes[2].set_title(f"Layer Depth Distribution of Top {TOP_N} Candidates")
axes[2].legend()

plt.tight_layout()
Path("figures").mkdir(exist_ok=True)
plt.savefig("figures/ablation_candidates.pdf", bbox_inches="tight")

# Analyze nearest neighbors for each reference head

In [ ]:
# show which candidates appear for multiple reference heads
candidate_counts: Counter[str] = Counter()
for ref_head, neighbors in CANDIDATES.all_neighbors.items():
	for neighbor, dist in neighbors:
		candidate_counts[neighbor] += 1

print("Heads appearing in top-K for multiple reference heads:")
for head, count in candidate_counts.most_common(20):
	if count > 1:
		print(f"  {head}: appears {count} times")

# Get control heads for baseline comparison

In [ ]:
# get control heads (far from induction heads) for each model
CONTROL_HEADS: dict[str, list[str]] = {}
for model in CANDIDATES.candidates_by_model.keys():
	controls = get_control_heads(
		HEAD_DISTS, CANDIDATES, model, n_controls=5, method="far"
	)
	CONTROL_HEADS[model] = controls
	print(f"\n{model} control heads (far from induction):")
	for head in controls:
		print(f"  {head}")

# Save candidates for ablation study

In [ ]:
# save candidates to file
output_path: Path = PATH_BASE / "ablation" / "candidates.json"
CANDIDATES.save(output_path)
print(f"Saved candidates to {output_path}")

# also save as CSV for easy viewing
CANDIDATES_DF.write_csv(PATH_BASE / "ablation" / "candidates.csv")